# PSO Analysis for ATP Player Classification

Experimental study of the **Particle Swarm Optimization (PSO)** algorithm applied to hyperparameter optimization in classification models, using ATP player data (2010–2023).

In [ ]:
import pandas as pd
import numpy as np
import random
import csv
import time

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE


## 1. Data Loading and Exploration

Load the ATP dataset containing match statistics for ATP circuit players from 2010 to 2023.

In [ ]:
data = pd.read_csv("../data/2010_2023.csv", low_memory=False)
data.head()


### Exploratory Visualization

Distribution of key features against the ranking group.

In [ ]:
columns = [
    ('matches_played', 'blue'),
    ('wins', 'black'),
    ('losses', 'red'),
    ('tournaments_played', 'coral'),
    ('win_percentage', 'green'),
    ('avg_opponent_rank_loss', 'purple'),
    ('straight_sets_wins', 'orange'),
    ('matches_won_high_1st_serves', 'pink')
]

fig, axes = plt.subplots(2, 4, figsize=(12, 6), sharey=True)
fig.tight_layout(pad=1.0)
axes = axes.flatten()

for ax, (col, color) in zip(axes, columns):
    ax.scatter(data[col], data['ranking_group'], c=color, alpha=0.5)
    ax.set_xlabel(col.replace('_', ' ').capitalize())

plt.show()


## 2. Preprocessing

Feature selection and 80/20 train/test split. Note the class imbalance visible in the training set distribution.

In [ ]:
features = [
    "matches_played", "wins", "losses", "tournaments_played",
    "win_percentage", "avg_opponent_rank_loss",
    "straight_sets_wins", "matches_won_high_1st_serves"
]

X = data[features]
y = data['ranking_group']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

hplot = sns.histplot(y_train, color='blue', label='Ranking', linewidth=0, bins=100)
hplot.set_xlabel('Ranking', fontsize=12)
hplot.set_ylabel('Number of instances', fontsize=12)
sns.set(rc={'figure.figsize': (4, 2)})


### Class Balancing with SMOTE

Apply SMOTE (*Synthetic Minority Over-sampling Technique*) to correct the class imbalance in the training set.

In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

hplot = sns.histplot(y_train_balanced, color='blue', label='Ranking', linewidth=0, bins=100)
hplot.set_xlabel('Ranking', fontsize=12)
hplot.set_ylabel('Number of instances', fontsize=12)
sns.set(rc={'figure.figsize': (4, 2)})


### Target Encoding and Correlation Heatmap

Encode `ranking_group` as integers and visualize feature correlations.

In [ ]:
ranking_group_mapping = {
    "Top 10": 0,
    "Top 100": 1,
    "Top 250": 2
}

y_train = y_train.map(ranking_group_mapping)
y_test = y_test.map(ranking_group_mapping)

data_encoded = data.copy()
data_encoded["ranking_group"] = data_encoded["ranking_group"].map(ranking_group_mapping)

corr_cols = features + ['ranking_group']
sns.heatmap(data_encoded[corr_cols].corr(), annot=True)
plt.show()


### Re-balance with Encoded Labels

Apply SMOTE on the numerically encoded training set.

In [ ]:
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)


## 3. Hyperparameter Search with Grid Search (Baseline)

Use Grid Search with Nested Cross-Validation as a baseline to compare against PSO.

Models evaluated: Random Forest, SVM, Logistic Regression, and k-NN.

In [ ]:
models = {
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [10, 100, 500],
            "criterion": ["gini", "entropy", "log_loss"],
            "max_depth": [5, 10, 20],
            "max_features": ["sqrt", "log2", None]
        }
    },
    "SVM": {
        "estimator": SVC(random_state=42),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf", "sigmoid"],
            "degree": [2, 3, 4]
        }
    },
    "Logistic Regression": {
        "estimator": LogisticRegression(random_state=42, max_iter=1000),
        "params": {
            "solver": ["newton-cg", "lbfgs", "liblinear", "sag", "saga"],
            "penalty": ["l1", "l2"],
            "C": [0.01, 0.1, 1]
        }
    },
    "k-NN": {
        "estimator": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 10],
            "weights": ["uniform", "distance"],
            "algorithm": ["auto", "ball_tree", "kd_tree"]
        }
    }
}

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
nested_results = {}

for model_name, config in models.items():
    print(f"\nProcessing model: {model_name}...")
    inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['params'],
        scoring="f1_macro",
        cv=inner_cv,
        n_jobs=-1,
        verbose=1
    )
    scores = cross_val_score(
        grid_search, X_train_balanced, y_train_balanced,
        scoring="f1_macro", cv=outer_cv, n_jobs=-1
    )
    nested_results[model_name] = {
        "mean_score": np.mean(scores),
        "std_score": np.std(scores),
        "all_scores": scores
    }
    print(f"Average F1-Macro (Nested CV): {np.mean(scores):.4f} ± {np.std(scores):.4f}")

best_model_name = max(nested_results, key=lambda x: nested_results[x]['mean_score'])
print(f"\nBest model after Nested Cross-Validation: {best_model_name}")

best_model = GridSearchCV(
    estimator=models[best_model_name]['estimator'],
    param_grid=models[best_model_name]['params'],
    scoring="f1_macro",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)
best_model.fit(X_train_balanced, y_train_balanced)
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))


## 4. PSO Optimization

Implement Particle Swarm Optimization (PSO) to search for the best Random Forest hyperparameters.

Two variants are compared:
- **PSO without mutations**: standard version
- **PSO with mutations**: adds perturbations to escape local minima

### Shared Setup

In [ ]:
# Categorical hyperparameter mappings
criterion_map = {0: 'gini', 1: 'entropy', 2: 'log_loss'}
max_features_map = {0: 'sqrt', 1: 'log2', 2: None}

# Hyperparameter search space
hyperparameter_space = {
    "n_estimators_values": [10, 50, 100, 200, 500],
    "max_depth_values": [5, 10, 20],
    "criterion_values": [0, 1, 2],
    "min_samples_split_values": [2, 5, 10],
    "min_samples_leaf_values": [1, 2, 3, 4, 5],
    "min_weight_fraction_leaf_values": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
    "max_features_values": [0, 1, 2]
}


def objective_fn(x):
    """Objective function: returns 1 - f1_macro for a given hyperparameter vector x."""
    model = RandomForestClassifier(
        n_estimators=round(x[0]),
        max_depth=round(x[1]),
        criterion=criterion_map[round(x[2])],
        min_samples_split=round(x[3]),
        min_samples_leaf=round(x[4]),
        min_weight_fraction_leaf=round(x[5], 2),
        max_features=max_features_map[round(x[6])]
    )
    scores = cross_val_score(
        model, X_train_balanced, y_train_balanced,
        scoring="f1_macro", cv=5, n_jobs=-1
    )
    return 1 - scores.mean()


class Particle:
    """Represents a PSO swarm particle."""
    def __init__(self, d):
        self.position = np.repeat(None, d)
        self.velocity = np.zeros(d)
        self.fitness = None
        self.best_local_fitness = None
        self.best_local_position = None


def calculate_diversity(particles):
    """Computes swarm diversity as the mean standard deviation of particle positions."""
    positions = np.array([p.position for p in particles])
    return np.mean(np.std(positions, axis=0))


### Unified PSO Function

A single `run_pso` function runs the algorithm with or without mutations.
Set `use_mutations=True` to enable stagnation detection and particle perturbation.

In [ ]:
def run_pso(use_mutations, output_file):
    """Run PSO over the hyperparameter search space.

    Args:
        use_mutations (bool): If True, apply mutations when the swarm stagnates.
        output_file (str): Path to the CSV file where results are saved.
    """
    d = len(hyperparameter_space)

    # Search space bounds
    lower_bound = [v[0] for v in hyperparameter_space.values()]
    upper_bound = [v[-1] for v in hyperparameter_space.values()]

    # PSO constants
    c1, c2 = 2, 2
    max_iter = 30

    # Mutation parameters
    mutation_threshold = 50
    mutation_rate = 0.3
    stagnation_limit = 5
    mutation_strength = 0.5

    with open(output_file, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['s', 'iter', 'cost_fitness', 'iteration_time', 'best_global_position'])

        for s in range(1, 8):
            N = 2 * s + 1
            w = 0.9
            evolution = np.zeros(max_iter + 1)

            particles = np.array([Particle(d) for _ in range(N)])

            # Swarm initialization
            start_time = time.time()
            for i in range(N):
                particles[i].position = [np.random.uniform(lower_bound[j], upper_bound[j]) for j in range(d)]
                particles[i].fitness = objective_fn(particles[i].position)
                particles[i].best_local_fitness = particles[i].fitness
                particles[i].best_local_position = particles[i].position

                if i == 0:
                    best_global_fitness = particles[i].fitness
                    best_global_position = particles[i].position
                elif particles[i].fitness < best_global_fitness:
                    best_global_fitness = particles[i].fitness
                    best_global_position = particles[i].position

            iteration_time = time.time() - start_time
            evolution[0] = best_global_fitness
            print(f'Iteration 0: Cost(Fitness) = {1 - evolution[0]:.4f} | Time: {iteration_time:.4f}')
            writer.writerow([s, 0, f'{1 - evolution[0]:.4f}', f'{iteration_time:.4f}', str(best_global_position)])

            diversity_history = []
            stagnation_count = 0

            # Main loop
            for it in range(1, max_iter + 1):
                start_time = time.time()

                if use_mutations:
                    diversity = calculate_diversity(particles)
                    diversity_history.append(diversity)
                    if len(diversity_history) > stagnation_limit:
                        diversity_history.pop(0)
                        if max(diversity_history) - min(diversity_history) < 5:
                            stagnation_count += 1
                        else:
                            stagnation_count = 0

                    if diversity < mutation_threshold and it > 5 and stagnation_count >= stagnation_limit:
                        num_mutations = max(1, int(mutation_rate * len(particles)))
                        mutated_indices = random.sample(range(len(particles)), num_mutations)
                        print(f'Mutation applied at iteration {it} to {num_mutations} particles.')
                        for idx in mutated_indices:
                            perturbation = [
                                np.random.uniform(-mutation_strength, mutation_strength) * (upper_bound[j] - lower_bound[j])
                                for j in range(d)
                            ]
                            particles[idx].position = np.clip(
                                [particles[idx].position[j] + perturbation[j] for j in range(d)],
                                lower_bound, upper_bound
                            )
                            particles[idx].fitness = objective_fn(particles[idx].position)
                            particles[idx].best_local_fitness = particles[idx].fitness
                            particles[idx].best_local_position = particles[idx].position
                        stagnation_count = 0

                # Update position and velocity of each particle
                for i in range(N):
                    position_arr = np.array(particles[i].position)
                    best_local_pos_arr = np.array(particles[i].best_local_position)
                    best_global_pos_arr = np.array(best_global_position)

                    w = w + 0.5 * np.exp(-np.abs(position_arr - best_local_pos_arr))
                    cognitive = c1 * random.uniform(0, 1) * (best_local_pos_arr - position_arr)
                    social = c2 * random.uniform(0, 1) * (best_global_pos_arr - position_arr)
                    particles[i].velocity = w * particles[i].velocity + cognitive + social
                    particles[i].position = particles[i].position + particles[i].velocity

                    # Enforce position bounds
                    for j in range(d):
                        if particles[i].position[j] < lower_bound[j]:
                            particles[i].position[j] = lower_bound[j]
                            particles[i].velocity[j] = -particles[i].velocity[j]
                        elif particles[i].position[j] > upper_bound[j]:
                            particles[i].position[j] = upper_bound[j]
                            particles[i].velocity[j] = -particles[i].velocity[j]

                    particles[i].fitness = objective_fn(particles[i].position)

                    # Update local best
                    if particles[i].fitness < particles[i].best_local_fitness:
                        particles[i].best_local_fitness = particles[i].fitness
                        particles[i].best_local_position = particles[i].position

                    # Update global best
                    if particles[i].best_local_fitness < best_global_fitness:
                        best_global_fitness = particles[i].best_local_fitness
                        best_global_position = particles[i].best_local_position

                evolution[it] = best_global_fitness
                iteration_time = time.time() - start_time
                cost_fitness = 1 - evolution[it]
                print(f'Iteration: {it} | Cost(Fitness) = {cost_fitness:.4f} | Time: {iteration_time:.4f}')
                writer.writerow([s, it, f'{cost_fitness:.4f}', f'{iteration_time:.4f}', str(best_global_position)])

        print('Final best global position:', best_global_position)


### PSO Without Mutations

In [ ]:
run_pso(use_mutations=False, output_file="results_no_mutations.csv")


### PSO With Mutations

In [ ]:
run_pso(use_mutations=True, output_file="results_with_mutations.csv")
